## Cleaning Data: making a DataFrame easier to work with

In [152]:
import pandas as pd 
import numpy as np 

df = pd.DataFrame(
    {
        "From_To": [
            "LoNDon_paris",
            "MAdrid_miLAN",
            "londON_StockhOlm",
            "Budapest_PaRis",
            "Brussels_londOn",
        ],
        "FlightNumber": [10045, np.nan, 10065, np.nan, 10085],
        "RecentDelays": [[23, 47], [], [24, 43, 87], [13], [67, 32]],
        "Airline": [
            "KLM(!)",
            "<Air France> (12)",
            "(British Airways. )",
            "12. Air France",
            '"Swiss Air"',
        ],
    }
)

df 

,From_To,FlightNumber,RecentDelays,Airline
0,LoNDon_paris,10045.0,"[23, 47]",KLM(!)
1,MAdrid_miLAN,NaN,[],<Air France> (12)
2,londON_StockhOlm,10065.0,"[24, 43, 87]",(British Airways. )
3,Budapest_PaRis,NaN,[13],12. Air France
4,Brussels_londOn,10085.0,"[67, 32]","""Swiss Air"""


**38.** Some values in the the **FlightNumber** column are missing (they are `NaN`). These numbers are meant to increase by 10 with each row so 10055 and 10075 need to be put in place. Modify `df` to fill in these missing numbers and make the column an integer column (instead of a float column).

In [153]:
df.FlightNumber = df.FlightNumber.where(df.FlightNumber.notna(), df.FlightNumber.shift() + 10).astype('int64')
df 

,From_To,FlightNumber,RecentDelays,Airline
0,LoNDon_paris,10045,"[23, 47]",KLM(!)
1,MAdrid_miLAN,10055,[],<Air France> (12)
2,londON_StockhOlm,10065,"[24, 43, 87]",(British Airways. )
3,Budapest_PaRis,10075,[13],12. Air France
4,Brussels_londOn,10085,"[67, 32]","""Swiss Air"""


**39.** The **From\_To** column would be better as two separate columns! Split each string on the underscore delimiter `_` to give a new temporary DataFrame called 'temp' with the correct values. Assign the correct column names 'From' and 'To' to this temporary DataFrame. 

In [154]:
temp = df.From_To.str.split('_', expand=True)
temp.columns = ['From', 'To']
temp 

,From,To
0,LoNDon,paris
1,MAdrid,miLAN
2,londON,StockhOlm
3,Budapest,PaRis
4,Brussels,londOn


**40.** Notice how the capitalisation of the city names is all mixed up in this temporary DataFrame 'temp'. Standardise the strings so that only the first letter is uppercase (e.g. "londON" should become "London".)

In [155]:
temp = temp.map(lambda val: val.capitalize())
temp 

,From,To
0,London,Paris
1,Madrid,Milan
2,London,Stockholm
3,Budapest,Paris
4,Brussels,London


**41.** Delete the **From_To** column from `df` and attach the temporary DataFrame 'temp' from the previous questions.

In [156]:
df = df.drop(columns='From_To')
df = pd.concat([temp, df], axis=1)
df 

,From,To,FlightNumber,RecentDelays,Airline
0,London,Paris,10045,"[23, 47]",KLM(!)
1,Madrid,Milan,10055,[],<Air France> (12)
2,London,Stockholm,10065,"[24, 43, 87]",(British Airways. )
3,Budapest,Paris,10075,[13],12. Air France
4,Brussels,London,10085,"[67, 32]","""Swiss Air"""


**42**. In the **Airline** column, you can see some extra puctuation and symbols have appeared around the airline names. Pull out just the airline name. E.g. `'(British Airways. )'` should become `'British Airways'`.

In [157]:
df.Airline = df.Airline.str.extract(r'([A-Za-z ]+)')
df.Airline = df.Airline.str.strip()
df 

,From,To,FlightNumber,RecentDelays,Airline
0,London,Paris,10045,"[23, 47]",KLM
1,Madrid,Milan,10055,[],Air France
2,London,Stockholm,10065,"[24, 43, 87]",British Airways
3,Budapest,Paris,10075,[13],Air France
4,Brussels,London,10085,"[67, 32]",Swiss Air


**43**. In the RecentDelays column, the values have been entered into the DataFrame as a list. We would like each first value in its own column, each second value in its own column, and so on. If there isn't an Nth value, the value should be NaN.

Expand the Series of lists into a DataFrame named `delays`, rename the columns `delay_1`, `delay_2`, etc. and replace the unwanted RecentDelays column in `df` with `delays`.

In [158]:
max_length = df.RecentDelays.apply(lambda rd: len(rd)).max()  
delays = df.RecentDelays.apply(lambda rd: rd + [np.nan] * (max_length - len(rd))).apply(pd.Series)
delays.columns = [f'delay_{n}' for n in range(1, max_length +1)]

df = df.drop(columns='RecentDelays')
df = pd.concat([df, delays], axis=1)
df = df[['From', 'To', 'FlightNumber'] + delays.columns.to_list() + ['Airline']]
df 

,From,To,FlightNumber,delay_1,delay_2,delay_3,Airline
0,London,Paris,10045,23.0,47.0,NaN,KLM
1,Madrid,Milan,10055,NaN,NaN,NaN,Air France
2,London,Stockholm,10065,24.0,43.0,87.0,British Airways
3,Budapest,Paris,10075,13.0,NaN,NaN,Air France
4,Brussels,London,10085,67.0,32.0,NaN,Swiss Air
